### Prepare machine-learning-based classification approach using labeled snippets
Author: Charlotte Bez

Date: March 2025

In [ ]:
import matplotlib.pyplot as plt
import numpy as n
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
news_articles_df = pd.read_csv("./Input_Few_Shot_Prompt.csv")

total_articles = news_articles_df.shape[0]
print("\nNumber of articles:", total_articles)

news_articles_df.head()

news_articles_df.isnull().sum() 
print(news_articles_df.isnull().sum())

news_articles_df['combined_text'] = news_articles_df['Headline'].map(str) +" "+ news_articles_df['Article'].map(str) 
news_articles_df['Count_Transition'] = news_articles_df['Article'].str.count(r'\b[tT]ransition\b')
news_articles_df.head()


Number of articles: 2281
ID                         0
Source_File                0
Newspaper                252
Date                       0
Length                     0
Section                  633
Author                   874
Edition                 2281
Headline                   0
Graphic                    0
ArticleCount               0
RelativeShare              0
Type                     304
Classification_topic     285
Article                    0
dtype: int64


ID                                                           Source_File  \
0   1  0 Data/LexisNexis/Scraper_Just_Transitions_SA/Merged/Merged_All.docx   
1   2  0 Data/LexisNexis/Scraper_Just_Transitions_SA/Merged/Merged_All.docx   
2   3  0 Data/LexisNexis/Scraper_Just_Transitions_SA/Merged/Merged_All.docx   
3   4  0 Data/LexisNexis/Scraper_Just_Transitions_SA/Merged/Merged_All.docx   
4   5  0 Data/LexisNexis/Scraper_Just_Transitions_SA/Merged/Merged_All.docx   

                       Newspaper  Date      Length  \
0                   Bizcommunity  2022   933 words   
1                   Bizcommunity  2022   851 words   
2  Daily Dispatch (South Africa)  2022  1176 words   
3    Business Day (South Africa)  2018   669 words   
4                Mail & Guardian  2021   768 words   

                              Section  \
0                              TRENDS   
1                      CLIMATE CHANGE   
2  ECONOMY, BUSINESS & FINANCE; Pg. 8   
3                 OPINION & EDITORIAL   
4                                 NaN   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      Author  \
0  Lethabo ManamelaLethabo Manamela took over as interim CEO of the South African National Energy Development Institute (Sanedi) just as the Covid-19 lockdown started in March 2020, but she hasn't allowed the pandemic to get her down. She was recently presented with the South African National Energy Association (SANEA) Premier Award as the energy sector's 2021 Shapeshifter in recognition of the state-owned Sanedi's achievement in meeting 97% of its annual targets despite the pandemic's disruptions.https://www.bizcommunity.com/Profile.aspx?pro=1910091   
1                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        NaN   
2                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                Kate Okorie   
3                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              NEVA MAKGETLA   
4                                                                                                                                                           

In [ ]:
date_df = pd.read_excel("./date_to_merge.xlsx")

merged_df = news_articles_df.merge(date_df, on=["Length", "Headline"], how="left")  

if len(merged_df) == len(news_articles_df):
    print("Merge successful: No rows lost.")
else:
    print(f"Merge resulted in a row count change: {len(news_articles_df)} -> {len(merged_df)}")

print(merged_df.head(1))

news_articles_df = merged_df

Merge resulted in a row count change: 2281 -> 2295
   ID_x                                                           Source_File  \
0     1  0 Data/LexisNexis/Scraper_Just_Transitions_SA/Merged/Merged_All.docx   

      Newspaper  Date_x     Length Section  \
0  Bizcommunity    2022  933 words  TRENDS   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      Author  \
0  Lethabo ManamelaLethabo Manamela took over as interim CEO of the South African National Energy Development Institute (Sanedi) just

In [ ]:
news_articles_df['Date_y'] = pd.to_datetime(news_articles_df['Date_y'], errors='coerce')

def generate_random_date(year):
    if pd.notna(year):
        month = np.random.randint(1, 13)
        day = np.random.randint(1, 29)
        return pd.Timestamp(year=int(year), month=month, day=day)
    return pd.NaT

news_articles_df['Date_x'] = news_articles_df['Date_x'].astype(str).apply(generate_random_date)

news_articles_df['Date_y'].fillna(news_articles_df['Date_x'], inplace=True)

news_articles_df.drop(columns=['Date_x'], inplace=True, errors='ignore')

news_articles_df['Date_y'] = news_articles_df['Date_y'].dt.strftime('%-d %B %Y')

news_articles_df.rename(columns={'Date_y': 'Date'}, inplace=True)

KeyError: 'Date_y'

In [ ]:
print(news_articles_df['Date'].head())

0    2022-01-05 00:00:00
1    2022-07-13 00:00:00
2    2022-11-30 00:00:00
3    2018-10-09 00:00:00
4    2021-01-25 00:00:00
Name: Date, dtype: object


In [ ]:
missing_dates = news_articles_df[news_articles_df["Date"].isna()]
print(f"Total rows with missing 'Date': {len(missing_dates)}")

print(f"Length of news_articles_df: {len(news_articles_df)}")
print(f"Length of date_df: {len(date_df)}")

Total rows with missing 'Date': 0
Length of news_articles_df: 2295
Length of date_df: 2407


Text snippets (sentences) that deal with just transition.

In [ ]:
def extract_valid_snippets(article_text):
    sentences = re.split(r'(?<=[.!?])\s', article_text)
    valid_snippets = []

    for sentence in sentences:
        if re.search(r'\btransitions?\b', sentence, re.IGNORECASE):
            words = sentence.split()
            if len(words) >= 4:
                sentence = re.sub(r'<!--.*?-->', '', sentence)
                sentence = re.sub(r'â\(EURO\)~JUST TRANSITIONâ\(EURO\)\(TM\)', '', sentence, flags=re.IGNORECASE)  
                sentence = sentence.strip()
                
                if sentence:
                    valid_snippets.append(sentence)

    return valid_snippets

snippets_data = []

for index, row in news_articles_df.iterrows():
    snippets = extract_valid_snippets(str(row['Article']))
    for snippet in snippets:
        snippets_data.append({'ID': row['ID_x'], 'Date': row['Date'], 'Article_Snippet': snippet})

In [ ]:
snippets_df = pd.DataFrame(snippets_data)

print(f"Snippets before duplication removal: {len(snippets_df)}")

def remove_similar_snippets(df, threshold=0.85):
    if df.empty:
        return df
    
    vectorizer = TfidfVectorizer().fit_transform(df['Article_Snippet'])
    cosine_matrix = cosine_similarity(vectorizer)

    to_remove = set()
    for i in range(len(df)):
        for j in range(i + 1, len(df)):
            if j not in to_remove and cosine_matrix[i, j] > threshold:
                to_remove.add(j)

    return df.drop(df.index[list(to_remove)])

snippets_df = remove_similar_snippets(snippets_df)

print(f"Snippets after duplication removal: {len(snippets_df)}")

print(f"Saved {len(snippets_df)} snippets successfully.")

pd.set_option('display.max_colwidth', None)

snippets_df.to_excel("./Output_Few_Shot_Prompt.xlsx", index=False)

lengths = snippets_df['Article_Snippet'].str.len()
min_len, avg_len, max_len = lengths.min(), lengths.mean(), lengths.max()
print(f"Minimum Length: {min_len}\nAverage Length: {avg_len}\nMaximum Length: {max_len}")

Snippets before duplication removal: 6948
Snippets after duplication removal: 6363
Saved 6363 snippets successfully.
Minimum Length: 28
Average Length: 224.7037560898947
Maximum Length: 7977


In [ ]:
phrases = ["just transition", "energy transition", "just energy transition"]

phrase_counts = {phrase: snippets_df["Article_Snippet"].str.lower().str.count(phrase).sum() for phrase in phrases}

print(phrase_counts)

{'just transition': 2348, 'energy transition': 2189, 'just energy transition': 1449}


In [ ]:
mask_keep = (
    snippets_df["Article_Snippet"].str.contains("just energy transition", na=False) |
    snippets_df["Article_Snippet"].str.contains("just transition", na=False)
)

snippets_df = snippets_df[mask_keep]

num_snippets = snippets_df.shape[0]

print("Number of snippets:", num_snippets)

Number of snippets: 2747


In [ ]:
num_rows = snippets_df.shape[0]
num_na_date = snippets_df["Date"].isna().sum()

print(f"Total rows: {num_rows}")
print(f"Missing 'Date' values: {num_na_date}")

Total rows: 2747
Missing 'Date' values: 0


In [ ]:
snippets_df.to_excel("./Output_Few_Shot_Prompt.xlsx", index=False)